# COMPX525 A2 Synthetic Audio Generation

Generate planned synthetic training audio with AudioLDM2.


In [ ]:
from __future__ import annotations

import csv
import json
import subprocess
import sys
import time
from pathlib import Path

print("Installing generation dependencies")
if True:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "torch==2.5.1+cu121",
        "torchaudio==2.5.1+cu121",
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
    ])
    # Kaggle's preinstalled torchvision is pinned to the default torch build.
    # AudioLDM2 does not need torchvision, but transformers probes it during
    # CLAP imports; a mismatched torchvision can fail import with
    # "operator torchvision::nms does not exist".
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "diffusers>=0.35.1",
    "transformers",
    "accelerate",
    "scipy",
    "soundfile",
    "sentencepiece",
])

import numpy as np
import scipy.io.wavfile
import torch
from diffusers import AudioLDM2Pipeline
from transformers import GPT2LMHeadModel

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
PLAN_PATH = WORK_DIR / "synthetic_plan.csv"
PLAN_PATH.write_text('synthetic_id,primary_label,common_name,scientific_name,class_name,source_bucket,real_train_audio_count,soundscape_labeled_segment_count,target_floor,synthetic_index_for_label,seed,duration_seconds,split,prompt,negative_prompt,relative_audio_path,filter_status,filter_reason\nsyn_116570_000,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,0,525000,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_000.wav,planned,\nsyn_116570_001,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,1,525001,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_001.wav,planned,\nsyn_116570_002,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,2,525002,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_002.wav,planned,\nsyn_116570_003,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,3,525003,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_003.wav,planned,\nsyn_116570_004,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,4,525004,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_004.wav,planned,\nsyn_116570_005,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,5,525005,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_005.wav,planned,\nsyn_116570_006,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,6,525006,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_006.wav,planned,\nsyn_116570_007,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,7,525007,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_007.wav,planned,\nsyn_116570_008,116570,Southern Spectacled Caiman,Caiman yacare,Reptilia,train_audio,1,26,10,8,525008,5.0,train,"Five seconds of a single Southern Spectacled Caiman (Caiman yacare) Reptilia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/116570/syn_116570_008.wav,planned,\nsyn_1491113_000,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,0,525009,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_000.wav,planned,\nsyn_1491113_001,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,1,525010,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_001.wav,planned,\nsyn_1491113_002,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,2,525011,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_002.wav,planned,\nsyn_1491113_003,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,3,525012,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_003.wav,planned,\nsyn_1491113_004,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,4,525013,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_004.wav,planned,\nsyn_1491113_005,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,5,525014,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_005.wav,planned,\nsyn_1491113_006,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,6,525015,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_006.wav,planned,\nsyn_1491113_007,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,7,525016,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_007.wav,planned,\nsyn_1491113_008,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,8,525017,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_008.wav,planned,\nsyn_1491113_009,1491113,Guaraní leaf-litter frog,Adenomera guarani,Amphibia,soundscape_only,0,158,10,9,525018,5.0,train,"Five seconds of a single Guaraní leaf-litter frog (Adenomera guarani) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1491113/syn_1491113_009.wav,planned,\nsyn_1595929_000,1595929,Uruguay Harlequin Frog,Lysapsus limellum,Amphibia,train_audio,5,0,10,0,525019,5.0,train,"Five seconds of a single Uruguay Harlequin Frog (Lysapsus limellum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1595929/syn_1595929_000.wav,planned,\nsyn_1595929_001,1595929,Uruguay Harlequin Frog,Lysapsus limellum,Amphibia,train_audio,5,0,10,1,525020,5.0,train,"Five seconds of a single Uruguay Harlequin Frog (Lysapsus limellum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1595929/syn_1595929_001.wav,planned,\nsyn_1595929_002,1595929,Uruguay Harlequin Frog,Lysapsus limellum,Amphibia,train_audio,5,0,10,2,525021,5.0,train,"Five seconds of a single Uruguay Harlequin Frog (Lysapsus limellum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1595929/syn_1595929_002.wav,planned,\nsyn_1595929_003,1595929,Uruguay Harlequin Frog,Lysapsus limellum,Amphibia,train_audio,5,0,10,3,525022,5.0,train,"Five seconds of a single Uruguay Harlequin Frog (Lysapsus limellum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1595929/syn_1595929_003.wav,planned,\nsyn_1595929_004,1595929,Uruguay Harlequin Frog,Lysapsus limellum,Amphibia,train_audio,5,0,10,4,525023,5.0,train,"Five seconds of a single Uruguay Harlequin Frog (Lysapsus limellum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/1595929/syn_1595929_004.wav,planned,\nsyn_209233_000,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,0,525024,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_000.wav,planned,\nsyn_209233_001,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,1,525025,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_001.wav,planned,\nsyn_209233_002,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,2,525026,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_002.wav,planned,\nsyn_209233_003,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,3,525027,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_003.wav,planned,\nsyn_209233_004,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,4,525028,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_004.wav,planned,\nsyn_209233_005,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,5,525029,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_005.wav,planned,\nsyn_209233_006,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,6,525030,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_006.wav,planned,\nsyn_209233_007,209233,Feral Horse,Equus caballus,Mammalia,train_audio,2,0,10,7,525031,5.0,train,"Five seconds of a single Feral Horse (Equus caballus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/209233/syn_209233_007.wav,planned,\nsyn_22930_000,22930,Basin White-lipped Frog,Leptodactylus syphax,Amphibia,train_audio,6,0,10,0,525032,5.0,train,"Five seconds of a single Basin White-lipped Frog (Leptodactylus syphax) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22930/syn_22930_000.wav,planned,\nsyn_22930_001,22930,Basin White-lipped Frog,Leptodactylus syphax,Amphibia,train_audio,6,0,10,1,525033,5.0,train,"Five seconds of a single Basin White-lipped Frog (Leptodactylus syphax) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22930/syn_22930_001.wav,planned,\nsyn_22930_002,22930,Basin White-lipped Frog,Leptodactylus syphax,Amphibia,train_audio,6,0,10,2,525034,5.0,train,"Five seconds of a single Basin White-lipped Frog (Leptodactylus syphax) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22930/syn_22930_002.wav,planned,\nsyn_22930_003,22930,Basin White-lipped Frog,Leptodactylus syphax,Amphibia,train_audio,6,0,10,3,525035,5.0,train,"Five seconds of a single Basin White-lipped Frog (Leptodactylus syphax) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22930/syn_22930_003.wav,planned,\nsyn_22961_000,22961,Pointedbelly Frog,Leptodactylus podicipinus,Amphibia,train_audio,6,72,10,0,525036,5.0,train,"Five seconds of a single Pointedbelly Frog (Leptodactylus podicipinus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22961/syn_22961_000.wav,planned,\nsyn_22961_001,22961,Pointedbelly Frog,Leptodactylus podicipinus,Amphibia,train_audio,6,72,10,1,525037,5.0,train,"Five seconds of a single Pointedbelly Frog (Leptodactylus podicipinus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22961/syn_22961_001.wav,planned,\nsyn_22961_002,22961,Pointedbelly Frog,Leptodactylus podicipinus,Amphibia,train_audio,6,72,10,2,525038,5.0,train,"Five seconds of a single Pointedbelly Frog (Leptodactylus podicipinus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22961/syn_22961_002.wav,planned,\nsyn_22961_003,22961,Pointedbelly Frog,Leptodactylus podicipinus,Amphibia,train_audio,6,72,10,3,525039,5.0,train,"Five seconds of a single Pointedbelly Frog (Leptodactylus podicipinus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22961/syn_22961_003.wav,planned,\nsyn_22967_000,22967,Marbled White-lipped Frog,Leptodactylus elenae,Amphibia,train_audio,8,310,10,0,525040,5.0,train,"Five seconds of a single Marbled White-lipped Frog (Leptodactylus elenae) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22967/syn_22967_000.wav,planned,\nsyn_22967_001,22967,Marbled White-lipped Frog,Leptodactylus elenae,Amphibia,train_audio,8,310,10,1,525041,5.0,train,"Five seconds of a single Marbled White-lipped Frog (Leptodactylus elenae) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22967/syn_22967_001.wav,planned,\nsyn_22985_000,22985,Peter\'s Jungle Frog,Leptodactylus petersii,Amphibia,train_audio,6,0,10,0,525042,5.0,train,"Five seconds of a single Peter\'s Jungle Frog (Leptodactylus petersii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22985/syn_22985_000.wav,planned,\nsyn_22985_001,22985,Peter\'s Jungle Frog,Leptodactylus petersii,Amphibia,train_audio,6,0,10,1,525043,5.0,train,"Five seconds of a single Peter\'s Jungle Frog (Leptodactylus petersii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22985/syn_22985_001.wav,planned,\nsyn_22985_002,22985,Peter\'s Jungle Frog,Leptodactylus petersii,Amphibia,train_audio,6,0,10,2,525044,5.0,train,"Five seconds of a single Peter\'s Jungle Frog (Leptodactylus petersii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22985/syn_22985_002.wav,planned,\nsyn_22985_003,22985,Peter\'s Jungle Frog,Leptodactylus petersii,Amphibia,train_audio,6,0,10,3,525045,5.0,train,"Five seconds of a single Peter\'s Jungle Frog (Leptodactylus petersii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/22985/syn_22985_003.wav,planned,\nsyn_23150_000,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,0,525046,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_000.wav,planned,\nsyn_23150_001,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,1,525047,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_001.wav,planned,\nsyn_23150_002,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,2,525048,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_002.wav,planned,\nsyn_23150_003,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,3,525049,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_003.wav,planned,\nsyn_23150_004,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,4,525050,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_004.wav,planned,\nsyn_23150_005,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,5,525051,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_005.wav,planned,\nsyn_23150_006,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,6,525052,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_006.wav,planned,\nsyn_23150_007,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,7,525053,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_007.wav,planned,\nsyn_23150_008,23150,Central Dwarf Frog,Physalaemus centralis,Amphibia,train_audio,1,0,10,8,525054,5.0,train,"Five seconds of a single Central Dwarf Frog (Physalaemus centralis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23150/syn_23150_008.wav,planned,\nsyn_23154_000,23154,Bahia Dwarf Frog,Physalaemus albifrons,Amphibia,train_audio,5,0,10,0,525055,5.0,train,"Five seconds of a single Bahia Dwarf Frog (Physalaemus albifrons) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23154/syn_23154_000.wav,planned,\nsyn_23154_001,23154,Bahia Dwarf Frog,Physalaemus albifrons,Amphibia,train_audio,5,0,10,1,525056,5.0,train,"Five seconds of a single Bahia Dwarf Frog (Physalaemus albifrons) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23154/syn_23154_001.wav,planned,\nsyn_23154_002,23154,Bahia Dwarf Frog,Physalaemus albifrons,Amphibia,train_audio,5,0,10,2,525057,5.0,train,"Five seconds of a single Bahia Dwarf Frog (Physalaemus albifrons) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23154/syn_23154_002.wav,planned,\nsyn_23154_003,23154,Bahia Dwarf Frog,Physalaemus albifrons,Amphibia,train_audio,5,0,10,3,525058,5.0,train,"Five seconds of a single Bahia Dwarf Frog (Physalaemus albifrons) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23154/syn_23154_003.wav,planned,\nsyn_23154_004,23154,Bahia Dwarf Frog,Physalaemus albifrons,Amphibia,train_audio,5,0,10,4,525059,5.0,train,"Five seconds of a single Bahia Dwarf Frog (Physalaemus albifrons) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23154/syn_23154_004.wav,planned,\nsyn_23176_000,23176,Cope\'s Swamp Froglet,Pseudopaludicola mystacalis,Amphibia,train_audio,4,0,10,0,525060,5.0,train,"Five seconds of a single Cope\'s Swamp Froglet (Pseudopaludicola mystacalis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23176/syn_23176_000.wav,planned,\nsyn_23176_001,23176,Cope\'s Swamp Froglet,Pseudopaludicola mystacalis,Amphibia,train_audio,4,0,10,1,525061,5.0,train,"Five seconds of a single Cope\'s Swamp Froglet (Pseudopaludicola mystacalis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23176/syn_23176_001.wav,planned,\nsyn_23176_002,23176,Cope\'s Swamp Froglet,Pseudopaludicola mystacalis,Amphibia,train_audio,4,0,10,2,525062,5.0,train,"Five seconds of a single Cope\'s Swamp Froglet (Pseudopaludicola mystacalis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23176/syn_23176_002.wav,planned,\nsyn_23176_003,23176,Cope\'s Swamp Froglet,Pseudopaludicola mystacalis,Amphibia,train_audio,4,0,10,3,525063,5.0,train,"Five seconds of a single Cope\'s Swamp Froglet (Pseudopaludicola mystacalis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23176/syn_23176_003.wav,planned,\nsyn_23176_004,23176,Cope\'s Swamp Froglet,Pseudopaludicola mystacalis,Amphibia,train_audio,4,0,10,4,525064,5.0,train,"Five seconds of a single Cope\'s Swamp Froglet (Pseudopaludicola mystacalis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23176/syn_23176_004.wav,planned,\nsyn_23176_005,23176,Cope\'s Swamp Froglet,Pseudopaludicola mystacalis,Amphibia,train_audio,4,0,10,5,525065,5.0,train,"Five seconds of a single Cope\'s Swamp Froglet (Pseudopaludicola mystacalis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23176/syn_23176_005.wav,planned,\nsyn_23724_000,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,0,525066,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_000.wav,planned,\nsyn_23724_001,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,1,525067,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_001.wav,planned,\nsyn_23724_002,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,2,525068,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_002.wav,planned,\nsyn_23724_003,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,3,525069,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_003.wav,planned,\nsyn_23724_004,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,4,525070,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_004.wav,planned,\nsyn_23724_005,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,5,525071,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_005.wav,planned,\nsyn_23724_006,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,6,525072,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_006.wav,planned,\nsyn_23724_007,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,7,525073,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_007.wav,planned,\nsyn_23724_008,23724,Waxy Monkey Tree Frog,Phyllomedusa sauvagii,Amphibia,train_audio,1,0,10,8,525074,5.0,train,"Five seconds of a single Waxy Monkey Tree Frog (Phyllomedusa sauvagii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/23724/syn_23724_008.wav,planned,\nsyn_24321_000,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,0,525075,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_000.wav,planned,\nsyn_24321_001,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,1,525076,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_001.wav,planned,\nsyn_24321_002,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,2,525077,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_002.wav,planned,\nsyn_24321_003,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,3,525078,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_003.wav,planned,\nsyn_24321_004,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,4,525079,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_004.wav,planned,\nsyn_24321_005,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,5,525080,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_005.wav,planned,\nsyn_24321_006,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,6,525081,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_006.wav,planned,\nsyn_24321_007,24321,Mato Grosso Snouted Tree Frog,Scinax acuminatus,Amphibia,train_audio,2,344,10,7,525082,5.0,train,"Five seconds of a single Mato Grosso Snouted Tree Frog (Scinax acuminatus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/24321/syn_24321_007.wav,planned,\nsyn_25073_000,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,0,525083,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_000.wav,planned,\nsyn_25073_001,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,1,525084,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_001.wav,planned,\nsyn_25073_002,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,2,525085,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_002.wav,planned,\nsyn_25073_003,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,3,525086,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_003.wav,planned,\nsyn_25073_004,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,4,525087,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_004.wav,planned,\nsyn_25073_005,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,5,525088,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_005.wav,planned,\nsyn_25073_006,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,6,525089,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_006.wav,planned,\nsyn_25073_007,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,7,525090,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_007.wav,planned,\nsyn_25073_008,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,8,525091,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_008.wav,planned,\nsyn_25073_009,25073,Chiasmocleis mehelyi,Chiasmocleis mehelyi,Amphibia,soundscape_only,0,24,10,9,525092,5.0,train,"Five seconds of a single Chiasmocleis mehelyi Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25073/syn_25073_009.wav,planned,\nsyn_25214_000,25214,Muller\'s Termite Frog,Dermatonotus muelleri,Amphibia,train_audio,3,0,10,0,525093,5.0,train,"Five seconds of a single Muller\'s Termite Frog (Dermatonotus muelleri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25214/syn_25214_000.wav,planned,\nsyn_25214_001,25214,Muller\'s Termite Frog,Dermatonotus muelleri,Amphibia,train_audio,3,0,10,1,525094,5.0,train,"Five seconds of a single Muller\'s Termite Frog (Dermatonotus muelleri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25214/syn_25214_001.wav,planned,\nsyn_25214_002,25214,Muller\'s Termite Frog,Dermatonotus muelleri,Amphibia,train_audio,3,0,10,2,525095,5.0,train,"Five seconds of a single Muller\'s Termite Frog (Dermatonotus muelleri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25214/syn_25214_002.wav,planned,\nsyn_25214_003,25214,Muller\'s Termite Frog,Dermatonotus muelleri,Amphibia,train_audio,3,0,10,3,525096,5.0,train,"Five seconds of a single Muller\'s Termite Frog (Dermatonotus muelleri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25214/syn_25214_003.wav,planned,\nsyn_25214_004,25214,Muller\'s Termite Frog,Dermatonotus muelleri,Amphibia,train_audio,3,0,10,4,525097,5.0,train,"Five seconds of a single Muller\'s Termite Frog (Dermatonotus muelleri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25214/syn_25214_004.wav,planned,\nsyn_25214_005,25214,Muller\'s Termite Frog,Dermatonotus muelleri,Amphibia,train_audio,3,0,10,5,525098,5.0,train,"Five seconds of a single Muller\'s Termite Frog (Dermatonotus muelleri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25214/syn_25214_005.wav,planned,\nsyn_25214_006,25214,Muller\'s Termite Frog,Dermatonotus muelleri,Amphibia,train_audio,3,0,10,6,525099,5.0,train,"Five seconds of a single Muller\'s Termite Frog (Dermatonotus muelleri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/25214/syn_25214_006.wav,planned,\nsyn_47158son01_000,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,0,525100,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_000.wav,planned,\nsyn_47158son01_001,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,1,525101,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_001.wav,planned,\nsyn_47158son01_002,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,2,525102,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_002.wav,planned,\nsyn_47158son01_003,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,3,525103,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_003.wav,planned,\nsyn_47158son01_004,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,4,525104,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_004.wav,planned,\nsyn_47158son01_005,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,5,525105,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_005.wav,planned,\nsyn_47158son01_006,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,6,525106,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_006.wav,planned,\nsyn_47158son01_007,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,7,525107,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_007.wav,planned,\nsyn_47158son01_008,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,8,525108,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_008.wav,planned,\nsyn_47158son01_009,47158son01,Insect sonotype01,Insect son01,Insecta,soundscape_only,0,46,10,9,525109,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son01/syn_47158son01_009.wav,planned,\nsyn_47158son02_000,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,0,525110,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_000.wav,planned,\nsyn_47158son02_001,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,1,525111,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_001.wav,planned,\nsyn_47158son02_002,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,2,525112,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_002.wav,planned,\nsyn_47158son02_003,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,3,525113,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_003.wav,planned,\nsyn_47158son02_004,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,4,525114,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_004.wav,planned,\nsyn_47158son02_005,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,5,525115,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_005.wav,planned,\nsyn_47158son02_006,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,6,525116,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_006.wav,planned,\nsyn_47158son02_007,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,7,525117,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_007.wav,planned,\nsyn_47158son02_008,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,8,525118,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_008.wav,planned,\nsyn_47158son02_009,47158son02,Insect sonotype02,Insect son02,Insecta,soundscape_only,0,14,10,9,525119,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son02/syn_47158son02_009.wav,planned,\nsyn_47158son03_000,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,0,525120,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_000.wav,planned,\nsyn_47158son03_001,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,1,525121,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_001.wav,planned,\nsyn_47158son03_002,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,2,525122,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_002.wav,planned,\nsyn_47158son03_003,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,3,525123,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_003.wav,planned,\nsyn_47158son03_004,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,4,525124,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_004.wav,planned,\nsyn_47158son03_005,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,5,525125,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_005.wav,planned,\nsyn_47158son03_006,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,6,525126,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_006.wav,planned,\nsyn_47158son03_007,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,7,525127,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_007.wav,planned,\nsyn_47158son03_008,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,8,525128,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_008.wav,planned,\nsyn_47158son03_009,47158son03,Insect sonotype03,Insect son03,Insecta,soundscape_only,0,66,10,9,525129,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son03/syn_47158son03_009.wav,planned,\nsyn_47158son04_000,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,0,525130,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_000.wav,planned,\nsyn_47158son04_001,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,1,525131,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_001.wav,planned,\nsyn_47158son04_002,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,2,525132,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_002.wav,planned,\nsyn_47158son04_003,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,3,525133,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_003.wav,planned,\nsyn_47158son04_004,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,4,525134,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_004.wav,planned,\nsyn_47158son04_005,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,5,525135,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_005.wav,planned,\nsyn_47158son04_006,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,6,525136,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_006.wav,planned,\nsyn_47158son04_007,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,7,525137,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_007.wav,planned,\nsyn_47158son04_008,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,8,525138,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_008.wav,planned,\nsyn_47158son04_009,47158son04,Insect sonotype04,Insect son04,Insecta,soundscape_only,0,34,10,9,525139,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son04/syn_47158son04_009.wav,planned,\nsyn_47158son05_000,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,0,525140,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_000.wav,planned,\nsyn_47158son05_001,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,1,525141,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_001.wav,planned,\nsyn_47158son05_002,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,2,525142,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_002.wav,planned,\nsyn_47158son05_003,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,3,525143,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_003.wav,planned,\nsyn_47158son05_004,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,4,525144,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_004.wav,planned,\nsyn_47158son05_005,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,5,525145,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_005.wav,planned,\nsyn_47158son05_006,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,6,525146,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_006.wav,planned,\nsyn_47158son05_007,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,7,525147,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_007.wav,planned,\nsyn_47158son05_008,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,8,525148,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_008.wav,planned,\nsyn_47158son05_009,47158son05,Insect sonotype05,Insect son05,Insecta,soundscape_only,0,6,10,9,525149,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son05/syn_47158son05_009.wav,planned,\nsyn_47158son06_000,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,0,525150,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_000.wav,planned,\nsyn_47158son06_001,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,1,525151,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_001.wav,planned,\nsyn_47158son06_002,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,2,525152,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_002.wav,planned,\nsyn_47158son06_003,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,3,525153,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_003.wav,planned,\nsyn_47158son06_004,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,4,525154,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_004.wav,planned,\nsyn_47158son06_005,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,5,525155,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_005.wav,planned,\nsyn_47158son06_006,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,6,525156,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_006.wav,planned,\nsyn_47158son06_007,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,7,525157,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_007.wav,planned,\nsyn_47158son06_008,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,8,525158,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_008.wav,planned,\nsyn_47158son06_009,47158son06,Insect sonotype06,Insect son06,Insecta,soundscape_only,0,36,10,9,525159,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son06/syn_47158son06_009.wav,planned,\nsyn_47158son07_000,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,0,525160,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_000.wav,planned,\nsyn_47158son07_001,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,1,525161,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_001.wav,planned,\nsyn_47158son07_002,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,2,525162,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_002.wav,planned,\nsyn_47158son07_003,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,3,525163,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_003.wav,planned,\nsyn_47158son07_004,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,4,525164,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_004.wav,planned,\nsyn_47158son07_005,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,5,525165,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_005.wav,planned,\nsyn_47158son07_006,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,6,525166,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_006.wav,planned,\nsyn_47158son07_007,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,7,525167,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_007.wav,planned,\nsyn_47158son07_008,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,8,525168,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_008.wav,planned,\nsyn_47158son07_009,47158son07,Insect sonotype07,Insect son07,Insecta,soundscape_only,0,96,10,9,525169,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son07/syn_47158son07_009.wav,planned,\nsyn_47158son08_000,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,0,525170,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_000.wav,planned,\nsyn_47158son08_001,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,1,525171,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_001.wav,planned,\nsyn_47158son08_002,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,2,525172,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_002.wav,planned,\nsyn_47158son08_003,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,3,525173,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_003.wav,planned,\nsyn_47158son08_004,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,4,525174,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_004.wav,planned,\nsyn_47158son08_005,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,5,525175,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_005.wav,planned,\nsyn_47158son08_006,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,6,525176,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_006.wav,planned,\nsyn_47158son08_007,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,7,525177,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_007.wav,planned,\nsyn_47158son08_008,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,8,525178,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_008.wav,planned,\nsyn_47158son08_009,47158son08,Insect sonotype08,Insect son08,Insecta,soundscape_only,0,34,10,9,525179,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son08/syn_47158son08_009.wav,planned,\nsyn_47158son09_000,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,0,525180,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_000.wav,planned,\nsyn_47158son09_001,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,1,525181,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_001.wav,planned,\nsyn_47158son09_002,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,2,525182,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_002.wav,planned,\nsyn_47158son09_003,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,3,525183,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_003.wav,planned,\nsyn_47158son09_004,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,4,525184,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_004.wav,planned,\nsyn_47158son09_005,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,5,525185,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_005.wav,planned,\nsyn_47158son09_006,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,6,525186,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_006.wav,planned,\nsyn_47158son09_007,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,7,525187,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_007.wav,planned,\nsyn_47158son09_008,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,8,525188,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_008.wav,planned,\nsyn_47158son09_009,47158son09,Insect sonotype09,Insect son09,Insecta,soundscape_only,0,12,10,9,525189,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son09/syn_47158son09_009.wav,planned,\nsyn_47158son10_000,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,0,525190,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_000.wav,planned,\nsyn_47158son10_001,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,1,525191,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_001.wav,planned,\nsyn_47158son10_002,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,2,525192,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_002.wav,planned,\nsyn_47158son10_003,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,3,525193,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_003.wav,planned,\nsyn_47158son10_004,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,4,525194,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_004.wav,planned,\nsyn_47158son10_005,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,5,525195,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_005.wav,planned,\nsyn_47158son10_006,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,6,525196,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_006.wav,planned,\nsyn_47158son10_007,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,7,525197,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_007.wav,planned,\nsyn_47158son10_008,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,8,525198,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_008.wav,planned,\nsyn_47158son10_009,47158son10,Insect sonotype10,Insect son10,Insecta,soundscape_only,0,66,10,9,525199,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son10/syn_47158son10_009.wav,planned,\nsyn_47158son11_000,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,0,525200,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_000.wav,planned,\nsyn_47158son11_001,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,1,525201,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_001.wav,planned,\nsyn_47158son11_002,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,2,525202,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_002.wav,planned,\nsyn_47158son11_003,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,3,525203,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_003.wav,planned,\nsyn_47158son11_004,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,4,525204,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_004.wav,planned,\nsyn_47158son11_005,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,5,525205,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_005.wav,planned,\nsyn_47158son11_006,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,6,525206,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_006.wav,planned,\nsyn_47158son11_007,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,7,525207,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_007.wav,planned,\nsyn_47158son11_008,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,8,525208,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_008.wav,planned,\nsyn_47158son11_009,47158son11,Insect sonotype11,Insect son11,Insecta,soundscape_only,0,72,10,9,525209,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son11/syn_47158son11_009.wav,planned,\nsyn_47158son12_000,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,0,525210,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_000.wav,planned,\nsyn_47158son12_001,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,1,525211,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_001.wav,planned,\nsyn_47158son12_002,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,2,525212,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_002.wav,planned,\nsyn_47158son12_003,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,3,525213,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_003.wav,planned,\nsyn_47158son12_004,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,4,525214,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_004.wav,planned,\nsyn_47158son12_005,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,5,525215,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_005.wav,planned,\nsyn_47158son12_006,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,6,525216,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_006.wav,planned,\nsyn_47158son12_007,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,7,525217,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_007.wav,planned,\nsyn_47158son12_008,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,8,525218,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_008.wav,planned,\nsyn_47158son12_009,47158son12,Insect sonotype12,Insect son12,Insecta,soundscape_only,0,10,10,9,525219,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son12/syn_47158son12_009.wav,planned,\nsyn_47158son13_000,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,0,525220,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_000.wav,planned,\nsyn_47158son13_001,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,1,525221,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_001.wav,planned,\nsyn_47158son13_002,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,2,525222,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_002.wav,planned,\nsyn_47158son13_003,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,3,525223,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_003.wav,planned,\nsyn_47158son13_004,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,4,525224,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_004.wav,planned,\nsyn_47158son13_005,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,5,525225,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_005.wav,planned,\nsyn_47158son13_006,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,6,525226,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_006.wav,planned,\nsyn_47158son13_007,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,7,525227,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_007.wav,planned,\nsyn_47158son13_008,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,8,525228,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_008.wav,planned,\nsyn_47158son13_009,47158son13,Insect sonotype13,Insect son13,Insecta,soundscape_only,0,72,10,9,525229,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son13/syn_47158son13_009.wav,planned,\nsyn_47158son14_000,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,0,525230,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_000.wav,planned,\nsyn_47158son14_001,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,1,525231,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_001.wav,planned,\nsyn_47158son14_002,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,2,525232,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_002.wav,planned,\nsyn_47158son14_003,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,3,525233,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_003.wav,planned,\nsyn_47158son14_004,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,4,525234,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_004.wav,planned,\nsyn_47158son14_005,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,5,525235,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_005.wav,planned,\nsyn_47158son14_006,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,6,525236,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_006.wav,planned,\nsyn_47158son14_007,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,7,525237,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_007.wav,planned,\nsyn_47158son14_008,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,8,525238,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_008.wav,planned,\nsyn_47158son14_009,47158son14,Insect sonotype14,Insect son14,Insecta,soundscape_only,0,24,10,9,525239,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son14/syn_47158son14_009.wav,planned,\nsyn_47158son15_000,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,0,525240,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_000.wav,planned,\nsyn_47158son15_001,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,1,525241,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_001.wav,planned,\nsyn_47158son15_002,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,2,525242,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_002.wav,planned,\nsyn_47158son15_003,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,3,525243,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_003.wav,planned,\nsyn_47158son15_004,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,4,525244,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_004.wav,planned,\nsyn_47158son15_005,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,5,525245,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_005.wav,planned,\nsyn_47158son15_006,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,6,525246,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_006.wav,planned,\nsyn_47158son15_007,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,7,525247,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_007.wav,planned,\nsyn_47158son15_008,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,8,525248,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_008.wav,planned,\nsyn_47158son15_009,47158son15,Insect sonotype15,Insect son15,Insecta,soundscape_only,0,24,10,9,525249,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son15/syn_47158son15_009.wav,planned,\nsyn_47158son16_000,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,0,525250,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_000.wav,planned,\nsyn_47158son16_001,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,1,525251,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_001.wav,planned,\nsyn_47158son16_002,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,2,525252,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_002.wav,planned,\nsyn_47158son16_003,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,3,525253,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_003.wav,planned,\nsyn_47158son16_004,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,4,525254,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_004.wav,planned,\nsyn_47158son16_005,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,5,525255,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_005.wav,planned,\nsyn_47158son16_006,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,6,525256,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_006.wav,planned,\nsyn_47158son16_007,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,7,525257,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_007.wav,planned,\nsyn_47158son16_008,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,8,525258,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_008.wav,planned,\nsyn_47158son16_009,47158son16,Insect sonotype16,Insect son16,Insecta,soundscape_only,0,24,10,9,525259,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son16/syn_47158son16_009.wav,planned,\nsyn_47158son17_000,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,0,525260,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_000.wav,planned,\nsyn_47158son17_001,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,1,525261,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_001.wav,planned,\nsyn_47158son17_002,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,2,525262,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_002.wav,planned,\nsyn_47158son17_003,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,3,525263,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_003.wav,planned,\nsyn_47158son17_004,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,4,525264,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_004.wav,planned,\nsyn_47158son17_005,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,5,525265,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_005.wav,planned,\nsyn_47158son17_006,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,6,525266,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_006.wav,planned,\nsyn_47158son17_007,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,7,525267,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_007.wav,planned,\nsyn_47158son17_008,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,8,525268,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_008.wav,planned,\nsyn_47158son17_009,47158son17,Insect sonotype17,Insect son17,Insecta,soundscape_only,0,86,10,9,525269,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son17/syn_47158son17_009.wav,planned,\nsyn_47158son18_000,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,0,525270,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_000.wav,planned,\nsyn_47158son18_001,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,1,525271,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_001.wav,planned,\nsyn_47158son18_002,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,2,525272,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_002.wav,planned,\nsyn_47158son18_003,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,3,525273,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_003.wav,planned,\nsyn_47158son18_004,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,4,525274,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_004.wav,planned,\nsyn_47158son18_005,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,5,525275,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_005.wav,planned,\nsyn_47158son18_006,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,6,525276,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_006.wav,planned,\nsyn_47158son18_007,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,7,525277,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_007.wav,planned,\nsyn_47158son18_008,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,8,525278,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_008.wav,planned,\nsyn_47158son18_009,47158son18,Insect sonotype18,Insect son18,Insecta,soundscape_only,0,24,10,9,525279,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son18/syn_47158son18_009.wav,planned,\nsyn_47158son19_000,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,0,525280,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_000.wav,planned,\nsyn_47158son19_001,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,1,525281,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_001.wav,planned,\nsyn_47158son19_002,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,2,525282,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_002.wav,planned,\nsyn_47158son19_003,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,3,525283,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_003.wav,planned,\nsyn_47158son19_004,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,4,525284,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_004.wav,planned,\nsyn_47158son19_005,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,5,525285,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_005.wav,planned,\nsyn_47158son19_006,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,6,525286,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_006.wav,planned,\nsyn_47158son19_007,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,7,525287,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_007.wav,planned,\nsyn_47158son19_008,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,8,525288,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_008.wav,planned,\nsyn_47158son19_009,47158son19,Insect sonotype19,Insect son19,Insecta,soundscape_only,0,10,10,9,525289,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son19/syn_47158son19_009.wav,planned,\nsyn_47158son20_000,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,0,525290,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_000.wav,planned,\nsyn_47158son20_001,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,1,525291,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_001.wav,planned,\nsyn_47158son20_002,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,2,525292,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_002.wav,planned,\nsyn_47158son20_003,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,3,525293,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_003.wav,planned,\nsyn_47158son20_004,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,4,525294,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_004.wav,planned,\nsyn_47158son20_005,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,5,525295,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_005.wav,planned,\nsyn_47158son20_006,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,6,525296,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_006.wav,planned,\nsyn_47158son20_007,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,7,525297,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_007.wav,planned,\nsyn_47158son20_008,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,8,525298,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_008.wav,planned,\nsyn_47158son20_009,47158son20,Insect sonotype20,Insect son20,Insecta,soundscape_only,0,24,10,9,525299,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son20/syn_47158son20_009.wav,planned,\nsyn_47158son21_000,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,0,525300,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_000.wav,planned,\nsyn_47158son21_001,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,1,525301,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_001.wav,planned,\nsyn_47158son21_002,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,2,525302,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_002.wav,planned,\nsyn_47158son21_003,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,3,525303,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_003.wav,planned,\nsyn_47158son21_004,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,4,525304,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_004.wav,planned,\nsyn_47158son21_005,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,5,525305,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_005.wav,planned,\nsyn_47158son21_006,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,6,525306,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_006.wav,planned,\nsyn_47158son21_007,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,7,525307,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_007.wav,planned,\nsyn_47158son21_008,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,8,525308,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_008.wav,planned,\nsyn_47158son21_009,47158son21,Insect sonotype21,Insect son21,Insecta,soundscape_only,0,44,10,9,525309,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son21/syn_47158son21_009.wav,planned,\nsyn_47158son22_000,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,0,525310,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_000.wav,planned,\nsyn_47158son22_001,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,1,525311,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_001.wav,planned,\nsyn_47158son22_002,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,2,525312,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_002.wav,planned,\nsyn_47158son22_003,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,3,525313,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_003.wav,planned,\nsyn_47158son22_004,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,4,525314,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_004.wav,planned,\nsyn_47158son22_005,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,5,525315,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_005.wav,planned,\nsyn_47158son22_006,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,6,525316,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_006.wav,planned,\nsyn_47158son22_007,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,7,525317,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_007.wav,planned,\nsyn_47158son22_008,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,8,525318,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_008.wav,planned,\nsyn_47158son22_009,47158son22,Insect sonotype22,Insect son22,Insecta,soundscape_only,0,48,10,9,525319,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son22/syn_47158son22_009.wav,planned,\nsyn_47158son23_000,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,0,525320,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_000.wav,planned,\nsyn_47158son23_001,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,1,525321,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_001.wav,planned,\nsyn_47158son23_002,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,2,525322,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_002.wav,planned,\nsyn_47158son23_003,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,3,525323,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_003.wav,planned,\nsyn_47158son23_004,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,4,525324,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_004.wav,planned,\nsyn_47158son23_005,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,5,525325,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_005.wav,planned,\nsyn_47158son23_006,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,6,525326,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_006.wav,planned,\nsyn_47158son23_007,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,7,525327,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_007.wav,planned,\nsyn_47158son23_008,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,8,525328,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_008.wav,planned,\nsyn_47158son23_009,47158son23,Insect sonotype23,Insect son23,Insecta,soundscape_only,0,48,10,9,525329,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son23/syn_47158son23_009.wav,planned,\nsyn_47158son24_000,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,0,525330,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_000.wav,planned,\nsyn_47158son24_001,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,1,525331,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_001.wav,planned,\nsyn_47158son24_002,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,2,525332,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_002.wav,planned,\nsyn_47158son24_003,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,3,525333,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_003.wav,planned,\nsyn_47158son24_004,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,4,525334,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_004.wav,planned,\nsyn_47158son24_005,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,5,525335,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_005.wav,planned,\nsyn_47158son24_006,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,6,525336,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_006.wav,planned,\nsyn_47158son24_007,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,7,525337,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_007.wav,planned,\nsyn_47158son24_008,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,8,525338,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_008.wav,planned,\nsyn_47158son24_009,47158son24,Insect sonotype24,Insect son24,Insecta,soundscape_only,0,48,10,9,525339,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son24/syn_47158son24_009.wav,planned,\nsyn_47158son25_000,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,0,525340,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_000.wav,planned,\nsyn_47158son25_001,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,1,525341,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_001.wav,planned,\nsyn_47158son25_002,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,2,525342,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_002.wav,planned,\nsyn_47158son25_003,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,3,525343,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_003.wav,planned,\nsyn_47158son25_004,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,4,525344,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_004.wav,planned,\nsyn_47158son25_005,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,5,525345,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_005.wav,planned,\nsyn_47158son25_006,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,6,525346,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_006.wav,planned,\nsyn_47158son25_007,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,7,525347,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_007.wav,planned,\nsyn_47158son25_008,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,8,525348,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_008.wav,planned,\nsyn_47158son25_009,47158son25,Insect sonotype25,Insect son25,Insecta,soundscape_only,0,168,10,9,525349,5.0,train,"Five seconds of a single Insecta acoustic sonotype call recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/47158son25/syn_47158son25_009.wav,planned,\nsyn_476521_000,476521,Cuyaba Dwarf Frog,Physalaemus nattereri,Amphibia,train_audio,3,0,10,0,525350,5.0,train,"Five seconds of a single Cuyaba Dwarf Frog (Physalaemus nattereri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/476521/syn_476521_000.wav,planned,\nsyn_476521_001,476521,Cuyaba Dwarf Frog,Physalaemus nattereri,Amphibia,train_audio,3,0,10,1,525351,5.0,train,"Five seconds of a single Cuyaba Dwarf Frog (Physalaemus nattereri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/476521/syn_476521_001.wav,planned,\nsyn_476521_002,476521,Cuyaba Dwarf Frog,Physalaemus nattereri,Amphibia,train_audio,3,0,10,2,525352,5.0,train,"Five seconds of a single Cuyaba Dwarf Frog (Physalaemus nattereri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/476521/syn_476521_002.wav,planned,\nsyn_476521_003,476521,Cuyaba Dwarf Frog,Physalaemus nattereri,Amphibia,train_audio,3,0,10,3,525353,5.0,train,"Five seconds of a single Cuyaba Dwarf Frog (Physalaemus nattereri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/476521/syn_476521_003.wav,planned,\nsyn_476521_004,476521,Cuyaba Dwarf Frog,Physalaemus nattereri,Amphibia,train_audio,3,0,10,4,525354,5.0,train,"Five seconds of a single Cuyaba Dwarf Frog (Physalaemus nattereri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/476521/syn_476521_004.wav,planned,\nsyn_476521_005,476521,Cuyaba Dwarf Frog,Physalaemus nattereri,Amphibia,train_audio,3,0,10,5,525355,5.0,train,"Five seconds of a single Cuyaba Dwarf Frog (Physalaemus nattereri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/476521/syn_476521_005.wav,planned,\nsyn_476521_006,476521,Cuyaba Dwarf Frog,Physalaemus nattereri,Amphibia,train_audio,3,0,10,6,525356,5.0,train,"Five seconds of a single Cuyaba Dwarf Frog (Physalaemus nattereri) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/476521/syn_476521_006.wav,planned,\nsyn_516975_000,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,0,525357,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_000.wav,planned,\nsyn_516975_001,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,1,525358,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_001.wav,planned,\nsyn_516975_002,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,2,525359,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_002.wav,planned,\nsyn_516975_003,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,3,525360,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_003.wav,planned,\nsyn_516975_004,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,4,525361,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_004.wav,planned,\nsyn_516975_005,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,5,525362,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_005.wav,planned,\nsyn_516975_006,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,6,525363,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_006.wav,planned,\nsyn_516975_007,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,7,525364,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_007.wav,planned,\nsyn_516975_008,516975,Hooded Capuchin,Sapajus cay,Mammalia,train_audio,1,26,10,8,525365,5.0,train,"Five seconds of a single Hooded Capuchin (Sapajus cay) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/516975/syn_516975_008.wav,planned,\nsyn_517063_000,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,0,525366,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_000.wav,planned,\nsyn_517063_001,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,1,525367,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_001.wav,planned,\nsyn_517063_002,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,2,525368,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_002.wav,planned,\nsyn_517063_003,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,3,525369,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_003.wav,planned,\nsyn_517063_004,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,4,525370,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_004.wav,planned,\nsyn_517063_005,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,5,525371,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_005.wav,planned,\nsyn_517063_006,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,6,525372,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_006.wav,planned,\nsyn_517063_007,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,7,525373,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_007.wav,planned,\nsyn_517063_008,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,8,525374,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_008.wav,planned,\nsyn_517063_009,517063,Southern Orange-legged Leaf Frog,Pithecopus azureus,Amphibia,soundscape_only,0,626,10,9,525375,5.0,train,"Five seconds of a single Southern Orange-legged Leaf Frog (Pithecopus azureus) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/517063/syn_517063_009.wav,planned,\nsyn_555123_000,555123,Usina Tree Frog,Boana lundii,Amphibia,train_audio,3,0,10,0,525376,5.0,train,"Five seconds of a single Usina Tree Frog (Boana lundii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555123/syn_555123_000.wav,planned,\nsyn_555123_001,555123,Usina Tree Frog,Boana lundii,Amphibia,train_audio,3,0,10,1,525377,5.0,train,"Five seconds of a single Usina Tree Frog (Boana lundii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555123/syn_555123_001.wav,planned,\nsyn_555123_002,555123,Usina Tree Frog,Boana lundii,Amphibia,train_audio,3,0,10,2,525378,5.0,train,"Five seconds of a single Usina Tree Frog (Boana lundii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555123/syn_555123_002.wav,planned,\nsyn_555123_003,555123,Usina Tree Frog,Boana lundii,Amphibia,train_audio,3,0,10,3,525379,5.0,train,"Five seconds of a single Usina Tree Frog (Boana lundii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555123/syn_555123_003.wav,planned,\nsyn_555123_004,555123,Usina Tree Frog,Boana lundii,Amphibia,train_audio,3,0,10,4,525380,5.0,train,"Five seconds of a single Usina Tree Frog (Boana lundii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555123/syn_555123_004.wav,planned,\nsyn_555123_005,555123,Usina Tree Frog,Boana lundii,Amphibia,train_audio,3,0,10,5,525381,5.0,train,"Five seconds of a single Usina Tree Frog (Boana lundii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555123/syn_555123_005.wav,planned,\nsyn_555123_006,555123,Usina Tree Frog,Boana lundii,Amphibia,train_audio,3,0,10,6,525382,5.0,train,"Five seconds of a single Usina Tree Frog (Boana lundii) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555123/syn_555123_006.wav,planned,\nsyn_555145_000,555145,Polka-dot Tree Frog,Boana punctata,Amphibia,train_audio,9,0,10,0,525383,5.0,train,"Five seconds of a single Polka-dot Tree Frog (Boana punctata) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/555145/syn_555145_000.wav,planned,\nsyn_64898_000,64898,Yungas de la Paz Poison Frog,Ameerega picta,Amphibia,train_audio,3,0,10,0,525384,5.0,train,"Five seconds of a single Yungas de la Paz Poison Frog (Ameerega picta) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/64898/syn_64898_000.wav,planned,\nsyn_64898_001,64898,Yungas de la Paz Poison Frog,Ameerega picta,Amphibia,train_audio,3,0,10,1,525385,5.0,train,"Five seconds of a single Yungas de la Paz Poison Frog (Ameerega picta) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/64898/syn_64898_001.wav,planned,\nsyn_64898_002,64898,Yungas de la Paz Poison Frog,Ameerega picta,Amphibia,train_audio,3,0,10,2,525386,5.0,train,"Five seconds of a single Yungas de la Paz Poison Frog (Ameerega picta) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/64898/syn_64898_002.wav,planned,\nsyn_64898_003,64898,Yungas de la Paz Poison Frog,Ameerega picta,Amphibia,train_audio,3,0,10,3,525387,5.0,train,"Five seconds of a single Yungas de la Paz Poison Frog (Ameerega picta) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/64898/syn_64898_003.wav,planned,\nsyn_64898_004,64898,Yungas de la Paz Poison Frog,Ameerega picta,Amphibia,train_audio,3,0,10,4,525388,5.0,train,"Five seconds of a single Yungas de la Paz Poison Frog (Ameerega picta) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/64898/syn_64898_004.wav,planned,\nsyn_64898_005,64898,Yungas de la Paz Poison Frog,Ameerega picta,Amphibia,train_audio,3,0,10,5,525389,5.0,train,"Five seconds of a single Yungas de la Paz Poison Frog (Ameerega picta) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/64898/syn_64898_005.wav,planned,\nsyn_64898_006,64898,Yungas de la Paz Poison Frog,Ameerega picta,Amphibia,train_audio,3,0,10,6,525390,5.0,train,"Five seconds of a single Yungas de la Paz Poison Frog (Ameerega picta) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/64898/syn_64898_006.wav,planned,\nsyn_66971_000,66971,Paraguayan Swimming Frog,Pseudis platensis,Amphibia,train_audio,5,298,10,0,525391,5.0,train,"Five seconds of a single Paraguayan Swimming Frog (Pseudis platensis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/66971/syn_66971_000.wav,planned,\nsyn_66971_001,66971,Paraguayan Swimming Frog,Pseudis platensis,Amphibia,train_audio,5,298,10,1,525392,5.0,train,"Five seconds of a single Paraguayan Swimming Frog (Pseudis platensis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/66971/syn_66971_001.wav,planned,\nsyn_66971_002,66971,Paraguayan Swimming Frog,Pseudis platensis,Amphibia,train_audio,5,298,10,2,525393,5.0,train,"Five seconds of a single Paraguayan Swimming Frog (Pseudis platensis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/66971/syn_66971_002.wav,planned,\nsyn_66971_003,66971,Paraguayan Swimming Frog,Pseudis platensis,Amphibia,train_audio,5,298,10,3,525394,5.0,train,"Five seconds of a single Paraguayan Swimming Frog (Pseudis platensis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/66971/syn_66971_003.wav,planned,\nsyn_66971_004,66971,Paraguayan Swimming Frog,Pseudis platensis,Amphibia,train_audio,5,298,10,4,525395,5.0,train,"Five seconds of a single Paraguayan Swimming Frog (Pseudis platensis) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/66971/syn_66971_004.wav,planned,\nsyn_67252_000,67252,Milk Frog,Trachycephalus typhonius,Amphibia,train_audio,6,4,10,0,525396,5.0,train,"Five seconds of a single Milk Frog (Trachycephalus typhonius) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/67252/syn_67252_000.wav,planned,\nsyn_67252_001,67252,Milk Frog,Trachycephalus typhonius,Amphibia,train_audio,6,4,10,1,525397,5.0,train,"Five seconds of a single Milk Frog (Trachycephalus typhonius) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/67252/syn_67252_001.wav,planned,\nsyn_67252_002,67252,Milk Frog,Trachycephalus typhonius,Amphibia,train_audio,6,4,10,2,525398,5.0,train,"Five seconds of a single Milk Frog (Trachycephalus typhonius) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/67252/syn_67252_002.wav,planned,\nsyn_67252_003,67252,Milk Frog,Trachycephalus typhonius,Amphibia,train_audio,6,4,10,3,525399,5.0,train,"Five seconds of a single Milk Frog (Trachycephalus typhonius) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/67252/syn_67252_003.wav,planned,\nsyn_70711_000,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,0,525400,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_000.wav,planned,\nsyn_70711_001,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,1,525401,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_001.wav,planned,\nsyn_70711_002,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,2,525402,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_002.wav,planned,\nsyn_70711_003,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,3,525403,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_003.wav,planned,\nsyn_70711_004,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,4,525404,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_004.wav,planned,\nsyn_70711_005,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,5,525405,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_005.wav,planned,\nsyn_70711_006,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,6,525406,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_006.wav,planned,\nsyn_70711_007,70711,Cei\'s White-lipped Frog,Leptodactylus macrosternum,Amphibia,train_audio,2,0,10,7,525407,5.0,train,"Five seconds of a single Cei\'s White-lipped Frog (Leptodactylus macrosternum) Amphibia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/70711/syn_70711_007.wav,planned,\nsyn_738183_000,738183,White-coated Titi,Plecturocebus pallescens,Mammalia,train_audio,5,0,10,0,525408,5.0,train,"Five seconds of a single White-coated Titi (Plecturocebus pallescens) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/738183/syn_738183_000.wav,planned,\nsyn_738183_001,738183,White-coated Titi,Plecturocebus pallescens,Mammalia,train_audio,5,0,10,1,525409,5.0,train,"Five seconds of a single White-coated Titi (Plecturocebus pallescens) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/738183/syn_738183_001.wav,planned,\nsyn_738183_002,738183,White-coated Titi,Plecturocebus pallescens,Mammalia,train_audio,5,0,10,2,525410,5.0,train,"Five seconds of a single White-coated Titi (Plecturocebus pallescens) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/738183/syn_738183_002.wav,planned,\nsyn_738183_003,738183,White-coated Titi,Plecturocebus pallescens,Mammalia,train_audio,5,0,10,3,525411,5.0,train,"Five seconds of a single White-coated Titi (Plecturocebus pallescens) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/738183/syn_738183_003.wav,planned,\nsyn_738183_004,738183,White-coated Titi,Plecturocebus pallescens,Mammalia,train_audio,5,0,10,4,525412,5.0,train,"Five seconds of a single White-coated Titi (Plecturocebus pallescens) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/738183/syn_738183_004.wav,planned,\nsyn_74580_000,74580,Black-tailed Marmoset,Mico melanurus,Mammalia,train_audio,3,0,10,0,525413,5.0,train,"Five seconds of a single Black-tailed Marmoset (Mico melanurus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/74580/syn_74580_000.wav,planned,\nsyn_74580_001,74580,Black-tailed Marmoset,Mico melanurus,Mammalia,train_audio,3,0,10,1,525414,5.0,train,"Five seconds of a single Black-tailed Marmoset (Mico melanurus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/74580/syn_74580_001.wav,planned,\nsyn_74580_002,74580,Black-tailed Marmoset,Mico melanurus,Mammalia,train_audio,3,0,10,2,525415,5.0,train,"Five seconds of a single Black-tailed Marmoset (Mico melanurus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/74580/syn_74580_002.wav,planned,\nsyn_74580_003,74580,Black-tailed Marmoset,Mico melanurus,Mammalia,train_audio,3,0,10,3,525416,5.0,train,"Five seconds of a single Black-tailed Marmoset (Mico melanurus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/74580/syn_74580_003.wav,planned,\nsyn_74580_004,74580,Black-tailed Marmoset,Mico melanurus,Mammalia,train_audio,3,0,10,4,525417,5.0,train,"Five seconds of a single Black-tailed Marmoset (Mico melanurus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/74580/syn_74580_004.wav,planned,\nsyn_74580_005,74580,Black-tailed Marmoset,Mico melanurus,Mammalia,train_audio,3,0,10,5,525418,5.0,train,"Five seconds of a single Black-tailed Marmoset (Mico melanurus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/74580/syn_74580_005.wav,planned,\nsyn_74580_006,74580,Black-tailed Marmoset,Mico melanurus,Mammalia,train_audio,3,0,10,6,525419,5.0,train,"Five seconds of a single Black-tailed Marmoset (Mico melanurus) Mammalia vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/74580/syn_74580_006.wav,planned,\nsyn_760266_000,760266,Prionacris erosa,Prionacris erosa,Insecta,train_audio,7,0,10,0,525420,5.0,train,"Five seconds of a single Prionacris erosa Insecta vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/760266/syn_760266_000.wav,planned,\nsyn_760266_001,760266,Prionacris erosa,Prionacris erosa,Insecta,train_audio,7,0,10,1,525421,5.0,train,"Five seconds of a single Prionacris erosa Insecta vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/760266/syn_760266_001.wav,planned,\nsyn_760266_002,760266,Prionacris erosa,Prionacris erosa,Insecta,train_audio,7,0,10,2,525422,5.0,train,"Five seconds of a single Prionacris erosa Insecta vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/760266/syn_760266_002.wav,planned,\nsyn_sptnig1_000,sptnig1,Spot-tailed Nightjar,Antiurus maculicaudus,Aves,train_audio,3,0,10,0,525423,5.0,train,"Five seconds of a single Spot-tailed Nightjar (Antiurus maculicaudus) Aves vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/sptnig1/syn_sptnig1_000.wav,planned,\nsyn_sptnig1_001,sptnig1,Spot-tailed Nightjar,Antiurus maculicaudus,Aves,train_audio,3,0,10,1,525424,5.0,train,"Five seconds of a single Spot-tailed Nightjar (Antiurus maculicaudus) Aves vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/sptnig1/syn_sptnig1_001.wav,planned,\nsyn_sptnig1_002,sptnig1,Spot-tailed Nightjar,Antiurus maculicaudus,Aves,train_audio,3,0,10,2,525425,5.0,train,"Five seconds of a single Spot-tailed Nightjar (Antiurus maculicaudus) Aves vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/sptnig1/syn_sptnig1_002.wav,planned,\nsyn_sptnig1_003,sptnig1,Spot-tailed Nightjar,Antiurus maculicaudus,Aves,train_audio,3,0,10,3,525426,5.0,train,"Five seconds of a single Spot-tailed Nightjar (Antiurus maculicaudus) Aves vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/sptnig1/syn_sptnig1_003.wav,planned,\nsyn_sptnig1_004,sptnig1,Spot-tailed Nightjar,Antiurus maculicaudus,Aves,train_audio,3,0,10,4,525427,5.0,train,"Five seconds of a single Spot-tailed Nightjar (Antiurus maculicaudus) Aves vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/sptnig1/syn_sptnig1_004.wav,planned,\nsyn_sptnig1_005,sptnig1,Spot-tailed Nightjar,Antiurus maculicaudus,Aves,train_audio,3,0,10,5,525428,5.0,train,"Five seconds of a single Spot-tailed Nightjar (Antiurus maculicaudus) Aves vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/sptnig1/syn_sptnig1_005.wav,planned,\nsyn_sptnig1_006,sptnig1,Spot-tailed Nightjar,Antiurus maculicaudus,Aves,train_audio,3,0,10,6,525429,5.0,train,"Five seconds of a single Spot-tailed Nightjar (Antiurus maculicaudus) Aves vocalization recorded in Brazil\'s Pantanal wetlands, clear foreground call, natural field-recording ambience, no human speech, no music.","human speech, music, microphone handling noise, clipping, distortion",synthetic_audio/sptnig1/syn_sptnig1_006.wav,planned,\n', encoding="utf-8")
OUTPUT_AUDIO_ROOT = WORK_DIR / "synthetic_audio"
OUTPUT_AUDIO_ROOT.mkdir(parents=True, exist_ok=True)

START = 0
LIMIT = 2
MODEL_ID = 'cvssp/audioldm2'
NUM_INFERENCE_STEPS = 25
GUIDANCE_SCALE = 3.5
AUDIO_LENGTH_IN_S = 5.0

with PLAN_PATH.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))

rows = all_rows[START:START + LIMIT]
print(json.dumps({
    "plan_rows_total": len(all_rows),
    "start": START,
    "limit": LIMIT,
    "selected_rows": len(rows),
    "model_id": MODEL_ID,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "audio_length_in_s": AUDIO_LENGTH_IN_S,
}, indent=2))

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()} device={device}")

language_model = GPT2LMHeadModel.from_pretrained(MODEL_ID, subfolder="language_model", torch_dtype=dtype)
pipe = AudioLDM2Pipeline.from_pretrained(MODEL_ID, torch_dtype=dtype, language_model=language_model)
if device == "cuda":
    try:
        pipe.enable_model_cpu_offload()
        generator_device = "cuda"
        print("Enabled model CPU offload")
    except Exception as exc:
        print(f"CPU offload unavailable: {exc}; using pipe.to(cuda)")
        pipe = pipe.to("cuda")
        generator_device = "cuda"
else:
    pipe = pipe.to("cpu")
    generator_device = "cpu"

generated_rows = []
started = time.time()

for n, row in enumerate(rows, start=1):
    item_started = time.time()
    seed = int(row["seed"])
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    prompt = row["prompt"]
    negative_prompt = row.get("negative_prompt") or None
    rel_path = Path(row["relative_audio_path"])
    out_path = WORK_DIR / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"[{n}/{len(rows)}] {row['synthetic_id']} label={row['primary_label']} seed={seed}")
    try:
        result = pipe(
            prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            audio_length_in_s=AUDIO_LENGTH_IN_S,
            num_waveforms_per_prompt=1,
            generator=generator,
        )
        audio = np.asarray(result.audios[0], dtype=np.float32)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
        audio = np.clip(audio, -1.0, 1.0)
        audio_int16 = (audio * 32767.0).astype(np.int16)
        scipy.io.wavfile.write(out_path, rate=16000, data=audio_int16)
        row["generated_audio_path"] = str(out_path)
        row["filter_status"] = "generated_pending_birdnet_filter"
        row["filter_reason"] = ""
        row["generation_elapsed_seconds"] = round(time.time() - item_started, 3)
    except Exception as exc:
        row["generated_audio_path"] = ""
        row["filter_status"] = "generation_failed"
        row["filter_reason"] = repr(exc)
        row["generation_elapsed_seconds"] = round(time.time() - item_started, 3)
        print(f"FAILED {row['synthetic_id']}: {exc!r}")
    generated_rows.append(row)

manifest_path = WORK_DIR / "synthetic_generation_manifest.csv"
fieldnames = list(generated_rows[0].keys()) if generated_rows else list(all_rows[0].keys())
with manifest_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(generated_rows)

summary = {
    "plan_rows_total": len(all_rows),
    "start": START,
    "limit": LIMIT,
    "selected_rows": len(rows),
    "generated_count": sum(r.get("filter_status") == "generated_pending_birdnet_filter" for r in generated_rows),
    "failed_count": sum(r.get("filter_status") == "generation_failed" for r in generated_rows),
    "elapsed_seconds": round(time.time() - started, 3),
    "model_id": MODEL_ID,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "audio_length_in_s": AUDIO_LENGTH_IN_S,
    "manifest_path": str(manifest_path),
}
(WORK_DIR / "synthetic_generation_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Synthetic generation complete")
